# Train E(n) Equivariant GNN on LP-PDBBind

This notebook trains the custom `GNNPredictor` architecture from the Virtual Lab Manager (VLM) repository on the LP-PDBBind dataset.

**Hardware:** Requires a GPU runtime (A100 recommended) to process 19K 3D molecular graphs and ESM-2 embeddings.

## 1. Environment Setup

In [ ]:
# Install dependencies (Colab has torch, we need PyG and RDKit)
!pip install -q torch-geometric rdkit transformers accelerate

## 2. Mount Google Drive, Clone Repo & Download Data

Everything is stored in Google Drive so your work persists across sessions.


In [ ]:
from google.colab import drive
import os, urllib.request

# Mount Google Drive for persistent storage
drive.mount('/content/drive')

# All work happens inside Drive so nothing is lost on timeout
PROJECT_DIR = '/content/drive/MyDrive/agentic-vlm'

if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/handeboyaci/agentic-vlm.git "$PROJECT_DIR"
else:
    print('Repo already cloned, pulling latest...')
    !cd "$PROJECT_DIR" && git pull

%cd "$PROJECT_DIR"

# Download LP-PDBBind dataset (~19K protein-ligand complexes)
os.makedirs('data/pdbbind_deepchem/raw', exist_ok=True)
csv_path = 'data/pdbbind_deepchem/raw/LP_PDBBind.csv'
if not os.path.exists(csv_path):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/THGLab/LP-PDBBind/master/dataset/LP_PDBBind.csv', csv_path)
    print('Downloaded LP_PDBBind.csv')
else:
    print('LP_PDBBind.csv already exists, skipping download.')


## 3. Dataset Preprocessing

In [ ]:
import torch
from torch_geometric.loader import DataLoader
from data.lp_pdbbind import LPPDBBind

# Uses cached processed data if available (skip re-processing)
# Delete data/pdbbind_deepchem/processed/ manually to force re-processing
train_full = LPPDBBind(root='data/pdbbind_deepchem', split='train')
test_dataset = LPPDBBind(root='data/pdbbind_deepchem', split='test')

# Carve out 10% of training set for validation (for early stopping)
train_full = train_full.shuffle()
val_size = int(0.1 * len(train_full))
val_dataset = train_full[:val_size]
train_dataset = train_full[val_size:]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')


## 4. Initialize the E(n) Equivariant Model

In [ ]:
from models.gnn_predictor import GNNPredictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}")

model = GNNPredictor(
    atom_feat_dim=43,    # RDKit atom features (11+7+6+8+5+6=43)
    hidden_dim=128,      # EGNN hidden dimension
    n_layers=4,          # Deep equivariant messaging
    dropout=0.2          # For MC Dropout later
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
criterion = torch.nn.MSELoss()

## 5. Training Loop
We train for 50 epochs, saving the best checkpoint based on validation loss.

In [ ]:
import copy
from tqdm.auto import tqdm

best_val_loss = float('inf')
best_model_weights = None
patience_counter = 0

EPOCHS = 100
PATIENCE = 15  # early stop if no improvement for 15 epochs

for epoch in range(1, EPOCHS + 1):
    # --- TRAIN --- 
    model.train()
    train_loss = 0
    num_train_graphs = 0
    for data in tqdm(train_loader, desc=f'Epoch {epoch} Train'):
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x.float(), data.pos.float(), data.batch, protein_embs=None)
        loss = criterion(out.squeeze(), data.y.float())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * data.num_graphs
        num_train_graphs += data.num_graphs
    
    train_loss /= num_train_graphs
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0
    num_val_graphs = 0
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            out = model(data.x.float(), data.pos.float(), data.batch, protein_embs=None)
            loss = criterion(out.squeeze(), data.y.float())
            val_loss += loss.item() * data.num_graphs
            num_val_graphs += data.num_graphs
            
    val_loss /= num_val_graphs
    scheduler.step(val_loss)
    
    print(f'Epoch {epoch:03d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}')
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
            break


## 6. Export Trained Weights

In [ ]:
import json as _json

# Save best model weights (already in Drive since we trained there)
torch.save(best_model_weights, 'gnn_predictor_trained.pth')
print('Saved best weights to gnn_predictor_trained.pth')

# Save training log to Drive
train_log = {
    'best_val_mse': best_val_loss,
    'epochs': EPOCHS,
    'model': 'GNNPredictor (E(n) Equivariant)',
    'atom_feat_dim': 65,
    'hidden_dim': 128,
    'n_layers': 4,
}
with open('training_log_egnn.json', 'w') as f:
    _json.dump(train_log, f, indent=2)

print(f'Best Val MSE: {best_val_loss:.4f}')
print('All outputs saved to Google Drive (persistent)!')
